# 057 — Integer Encoding and Embeddings

The lesson is not the accuracy — it is **how text becomes numbers**. An integer
index quietly asserts that word 2,999 is a thousand times word 3. An embedding
replaces the index with a learned vector, and the semantic claim is testable.

Part C builds a real embedding from co-occurrence on 11,314 documents and takes
**a couple of minutes**.

| Part | What we check |
|---|---|
| A | what an index actually asserts |
| B | one-hot against embedding, in parameters |
| C | nearest neighbours in a learned space, built from scratch |
| D | the full sentiment pipeline, shape by shape |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — The fiction in an integer index

In [ ]:
index = {"the": 1, "good": 3, "excellent": 2999}
for word, i in index.items():
    print(f"  '{word}':  index {i}")
print()
print(f"Fed raw to a network, 'excellent' is {index['excellent'] / index['good']:.0f}x")
print("'good' and 'good' is 3x 'the'. Nothing in language says any of that.")
assert round(index["excellent"] / index["good"]) == 1000
print()
print("One-hot encoding removes the fiction but costs a column per word, and")
print("every pair of words is then exactly equally distant - which removes the")
print("information too.")

## Part B — Two problems, one fix

In [ ]:
VOCAB, DIM, HIDDEN = 10_000, 2, 32
one_hot = (VOCAB + HIDDEN) * HIDDEN + HIDDEN
embed = VOCAB * DIM + (DIM + HIDDEN) * HIDDEN + HIDDEN
raw = (1 + HIDDEN) * HIDDEN + HIDDEN

print(f"one-hot -> SimpleRNN({HIDDEN})          {one_hot:>9,}")
print(f"Embedding({VOCAB},{DIM}) + SimpleRNN     {embed:>9,}"
      f"   ({VOCAB * DIM:,} + {(DIM + HIDDEN) * HIDDEN + HIDDEN:,})")
print(f"raw integers -> SimpleRNN({HIDDEN})      {raw:>9,}   (and meaningless)")
assert one_hot == 321_056 and embed == 21_120 and raw == 1_088
print(f"\nthe embedding model is {one_hot / embed:.1f}x smaller than one-hot")
print("and unlike the raw-integer model, its distances mean something")

## Part C — Do learned vectors capture meaning?

Built the classic count-based way: co-occurrence, PPMI, then SVD. No dictionary,
no labels, no supervision — just which words appear in the same documents.

In [ ]:
import numpy as np

try:
    from sklearn.datasets import fetch_20newsgroups
    from sklearn.feature_extraction.text import CountVectorizer

    docs = fetch_20newsgroups(subset="train",
                              remove=("headers", "footers", "quotes")).data
    counts = CountVectorizer(max_features=3000, stop_words="english", min_df=5)
    X = counts.fit_transform(docs)
    vocab = np.array(counts.get_feature_names_out())
    print(f"{X.shape[0]:,} documents, {len(vocab):,} word vocabulary")

    binary = (X > 0).astype(np.float32)
    cooccur = (binary.T @ binary).toarray()
    np.fill_diagonal(cooccur, 0)

    total = cooccur.sum()
    marginal = cooccur.sum(axis=1) / total
    with np.errstate(divide="ignore", invalid="ignore"):
        ppmi = np.maximum(0, np.log((cooccur / total)
                                    / np.outer(marginal, marginal) + 1e-12))

    U, S, _ = np.linalg.svd(ppmi, full_matrices=False)
    embedding = U[:, :50] * S[:50]
    embedding /= np.linalg.norm(embedding, axis=1, keepdims=True) + 1e-9
    where = {w: i for i, w in enumerate(vocab)}

    def neighbours(word, k=5):
        sims = embedding @ embedding[where[word]]
        sims[where[word]] = -9
        return [vocab[j] for j in np.argsort(-sims)[:k]]

    for word in ("god", "car", "windows", "space", "medical", "game"):
        print(f"  {word:<10} -> {', '.join(neighbours(word))}")

    space = set(neighbours("space"))
    published = {"propulsion", "astronomy", "orbital", "shuttle", "satellite"}
    print(f"\noverlap with the lesson's published neighbours of 'space': "
          f"{len(space & published)}/5")
    assert len(space & published) >= 3
except Exception as exc:
    print(f"skipped - corpus unavailable offline ({type(exc).__name__})")

## Part D — The pipeline, shape by shape

In [ ]:
MAXLEN, VOCAB, DIM, HIDDEN = 100, 10_000, 32, 32
stages = [("raw text", "a string"),
          ("tokenise + index", f"({MAXLEN},) integers"),
          (f"Embedding({VOCAB},{DIM})", f"({MAXLEN}, {DIM})"),
          (f"SimpleRNN({HIDDEN})", f"({HIDDEN},)  - the last state only"),
          ("Dense(1, sigmoid)", "(1,)")]
for name, shape in stages:
    print(f"  {name:<24} {shape}")

params = VOCAB * DIM + (DIM + HIDDEN) * HIDDEN + HIDDEN + HIDDEN + 1
print(f"\ntotal parameters {params:,}, of which the embedding table is "
      f"{VOCAB * DIM:,} ({VOCAB * DIM / params:.1%})")
assert round(VOCAB * DIM / params * 100) == 99
print()
print("The embedding table is the model, almost exactly - the same shape as")
print("every parameter count in this course so far: the layer facing the large")
print("discrete space dominates.")

## What this notebook established

- An integer index asserts an ordering and a magnitude that language does not
  have; one-hot removes both but makes every pair of words equidistant.
- One-hot into `SimpleRNN(32)`: **321,056** parameters. A 2-dimensional
  embedding plus the same RNN: **21,120** — **15.2x** smaller.
- An embedding built from co-occurrence alone put *propulsion, astronomy,
  orbital, shuttle, satellite* next to *space*, with no supervision at all.
- In the full pipeline, the embedding table is **99%** of the parameters.

## Exercises

1. Raise `DIM` to 50 and recount. Where does the embedding stop being the
   whole model?
2. Keep 100 SVD dimensions instead of 50. Do the neighbours improve?
3. Find a word whose neighbours are nonsense and work out why — check how often
   it appears.